# Introduction to Data Science 2025

# Week 4

In this week's exercise, we look at prompting and zero- and few-shot task settings. Below is a text generation example from https://github.com/TurkuNLP/intro-to-nlp/blob/master/text_generation_pipeline_example.ipynb demonstrating how to load a text generation pipeline with a pre-trained model and generate text with a given prompt. Your task is to load a similar pre-trained generative model and assess whether the model succeeds at a set of tasks in zero-shot, one-shot, and two-shot settings.

**Note: Downloading and running the pre-trained model locally may take some time. Alternatively, you can open and run this notebook on [Google Colab](https://colab.research.google.com/), as assumed in the following example.**

## Text generation example

This is a brief example of how to run text generation with a causal language model and `pipeline`.

Install [transformers](https://huggingface.co/docs/transformers/index) python package. This will be used to load the model and tokenizer and to run generation.

In [1]:
!pip install --quiet transformers
!pip install torch torchvision torchaudio
!pip3 install torch torchvision
!pip install pandas


  Using cached torch-2.8.0-cp313-none-macosx_11_0_arm64.whl.metadata (30 kB)
  Using cached torchvision-0.23.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (6.1 kB)
  Using cached torchaudio-2.8.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (7.2 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.5-py3-none-any.whl.metadata (6.3 kB)
  Using cached pillow-11.3.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (9.0 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached torch-2.8.0-cp313-none-macosx_11_0_arm64.whl (73.6 MB)
Using cached torchvision-0.23.0-cp313-cp313-macosx_11_0_arm64.whl (1.9 MB)
Using cached torchaudio-2.8.0-cp313-cp313-macosx_11_0_arm64.whl (1.9 MB)
Using cached pillow-11.3.0-cp313-cp313-macosx_11_0_arm64.whl (4.7 MB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
Using cached networkx-3.5-py3-none-any.whl (2.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Import the `AutoTokenizer`, `AutoModelForCausalLM`, and `pipeline` classes. The first two support loading tokenizers and generative models from the [Hugging Face repository](https://huggingface.co/models), and the last wraps a tokenizer and a model for convenience.

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline


/Users/nawara/Documents/exchange/introDS/exercises/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load a generative model and its tokenizer. You can substitute any other generative model name here (e.g. [other TurkuNLP GPT-3 models](https://huggingface.co/models?sort=downloads&search=turkunlp%2Fgpt3)), but note that Colab may have issues running larger models. 

In [3]:
MODEL_NAME = 'TurkuNLP/gpt3-finnish-large'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

model.safetensors:   0%|          | 0.00/3.25G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Instantiate a text generation pipeline using the tokenizer and model.

In [3]:
pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    device=model.device
)

Device set to use cpu


We can now call the pipeline with a text prompt; it will take care of tokenizing, encoding, generation, and decoding:

In [4]:
output = pipe('Terve, miten menee?', max_new_tokens=25)

print(output)

[{'generated_text': 'Terve, miten menee?”\n”Hyvin, kiitos.”\n”Kiva nähdä sinusta taas.”\n”Samat sanat.”\n”Nähdään'}]


Just print the text

In [5]:
print(output[0]['generated_text'])

Terve, miten menee?”
”Hyvin, kiitos.”
”Kiva nähdä sinusta taas.”
”Samat sanat.”
”Nähdään


We can also call the pipeline with any arguments that the model `generate` function supports. For details on text generation using `transformers`, see e.g. [this tutorial](https://huggingface.co/blog/how-to-generate).

Example with sampling and a high `temperature` parameter to generate more chaotic output:

In [6]:
output = pipe(
    'Terve, miten menee?',
    do_sample=True,
    temperature=10.0,
    max_new_tokens=25
)


print(output[0]['generated_text'])

Terve, miten menee? :d Tuli mieleen ihan kun tuossa muutama tovi jo kirjotellu mutten saa niitä kirjoitettua yhteen pötkytsköön


## Exercise 1

Your task is to assess whether a generative model succeeds in the following tasks in zero-shot, one-shot, and two-shot settings:

- binary sentiment classification (positive / negative)

- person name recognition

- two-digit addition (e.g. 11 + 22 = 33)

For example, for assessing whether a generative model can name capital cities, we could use the following prompts:

- zero-shot:
	>"""\
	>Identify the capital cities of countries.
	>
	>Question: What is the capital of Finland?\
	>Answer:\
	>"""
- one-shot:
	>"""\
	>Identify the capital cities of countries.
	>
	>Question: What is the capital of Sweden?\
	>Answer: Stockholm
	>
	>Question: What is the capital of Finland?\
	>Answer:\
	>"""
- two-shot:
	>"""\
	>Identify the capital cities of countries.
	>
	>Question: What is the capital of Sweden?\
	>Answer: Stockholm
	>
	>Question: What is the capital of Denmark?\
	>Answer: Copenhagen
	>
	>Question: What is the capital of Finland?\
	>Answer:\
	>"""

You can do the tasks either in English or Finnish and use a generative model of your choice from the Hugging Face models repository, for example the following models:

- English: `gpt2-large`
- Finnish: `TurkuNLP/gpt3-finnish-large`

You can either come up with your own instructions for the tasks or use the following:

- English:
	- binary sentiment classification: "Do the following texts express a positive or negative sentiment?"
	- person name recognition: "List the person names occurring in the following texts."
	- two-digit addition: "This is a first grade math exam."
- Finnish:
	- binary sentiment classification: "Ilmaisevatko seuraavat tekstit positiivista vai negatiivista tunnetta?"
	- person name recognition: "Listaa seuraavissa teksteissä mainitut henkilönnimet."
	- two-digit addition: "Tämä on ensimmäisen luokan matematiikan koe."

Come up with at least two test cases for each of the three tasks, and come up with your own one- and two-shot examples.

In [13]:
# Use this cell for your code

MODEL_NAME = 'gpt2-large'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)



In [15]:

pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    device=model.device
)

tests=["This movie is fantastic! i love it so much","This product is awful,I hate it so much"]
print("binary sentiment classification\n")
prompts={}
for i, text in enumerate(tests):
    prompts[f"test_{i+1}"] = {
        "zero": f"Do the following texts express a positive or negative sentiment?\nText: {text}\nAnswer: ",
        "one": (
            "Do the following texts express a positive or negative sentiment?\n"
            "Text: The weather is beautiful today!\nAnswer: positive\n"
            f"Text: {text}\nAnswer: "
        ),
        "two": (
            "Do the following texts express a positive or negative sentiment?\n"
            "Text: The weather is beautiful today!\nAnswer: positive\n"
            "Text: I hate driving during rush hour\nAnswer: negative\n"
            f"Text: {text}\nAnswer: "
        )
    }
for test, shot in prompts.items():
    print(f"\nResults for {test}:")
    for shot_type, prompt in shot.items():
        response = pipe(
            prompt,
            max_new_tokens=3,
            do_sample=False
        )[0]['generated_text']

        answer = response.split("Answer:")[-1].strip().split("\n")[0] #cleaner print, only getting whats after answer
        
        print(f"  In the {shot_type}-shot setting, the model predicted: {answer}")

#not very accurate... its supposed to be positive for test 1 and negative for test 2. 



Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


sentiment classification


Results for test_1:


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


  In the zero-shot setting, the model predicted: I don


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


...

  In the two-shot setting, the model predicted: negative

Results for test_2:


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


  In the zero-shot setting, the model predicted: I hate


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


  In the one-shot setting, the model predicted: negative
  In the two-shot setting, the model predicted: negative


**Submit this exercise by submitting your code and your answers to the above questions as comments on the MOOC platform. You can return this Jupyter notebook (.ipynb) or .py, .R, etc depending on your programming preferences.**

In [19]:
#person name recognition: "List the person names occurring in the following texts."
tests=["Barack Obama and Donald Trump were both presidents of the USA","Today, Angela and her boyfriend Mark went to the beach together"]
print("Person Name recognition\n")

prompts={}
for i, text in enumerate(tests):
    prompts[f"test_{i+1}"] = {
        "zero": f"List the person names occurring in the following texts.\nText: {text}\nAnswer: ",
        "one": (
            "List the person names occurring in the following texts.\n"
            "Text: My best friends are John and Mary.\nAnswer: John, Mary\n"
            f"Text: {text}\nAnswer: "
        ),
        "two": (
            "List the person names occurring in the following texts.\n"
            "Text: My best friends are John and Mary.\nAnswer: John, Mary\n"
            "Text: Angelica Bassett is the costar of Jonathan Bailey.\nAnswer: Angelica Bassett, Jonathan Bailey\n"
            f"Text: {text}\nAnswer: "
        )
    }

for test, shot in prompts.items():
    print(f"\nResults for {test}:")
    for shot_type, prompt in shot.items():
        response = pipe(
            prompt,
            max_new_tokens=15,
            do_sample=False
        )[0]['generated_text']

        answer = response.split("Answer:")[-1].strip().split("\n")[0]
        print(f" In the {shot_type}-shot setting, the model predicted: {answer}")
#pretty accurate in one and two shot settings, not so much zero shot

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Person Name recognition


Results for test_1:


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


 In the zero-shot setting, the model predicted: The first president of the USA was Abraham Lincoln.


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


...

 In the two-shot setting, the model predicted: Barack Obama, Donald Trump

Results for test_2:


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


 In the zero-shot setting, the model predicted: Mark


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


 In the one-shot setting, the model predicted: Mark, Angela
 In the two-shot setting, the model predicted: Angela, Mark


In [23]:
#two-digit addition: "This is a first grade math exam."

tests = ["3 + 2","14 + 9"]

print("two-digit addition\n")

prompts = {}
for i, text in enumerate(tests):
    prompts[f"test_{i+1}"] = {
        "zero": f"This is a first grade math exam.\nQuestion: {text}\nAnswer: ",
        "one": (
            "This is a first grade math exam.\n"
            "Question: 2 + 1\nAnswer: 3\n"
            f"Question: {text}\nAnswer: "
        ),
        "two": (
            "This is a first grade math exam.\n"
            "Question: 2 + 1\nAnswer: 3\n"
            "Question: 10 + 58\nAnswer: 68\n"
            f"Question: {text}\nAnswer: "
        )
    }

for test, shot in prompts.items():
    print(f"\nResults for {test}:")
    for shot_type, prompt in shot.items():
        response = pipe(
            prompt,
            max_new_tokens=5,
            do_sample=False
        )[0]['generated_text']

        answer = response.split("Answer:")[-1].strip().split("\n")[0]
        print(f"In the {shot_type}-shot setting, the model predicted: {answer}")


#pretty much failed at doing math questions

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


two-digit addition


Results for test_1:


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In the zero-shot setting, the model predicted: 3 + 2


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


...

In the two-shot setting, the model predicted: 3

Results for test_2:


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In the zero-shot setting, the model predicted: 14 + 9


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In the one-shot setting, the model predicted: 14
In the two-shot setting, the model predicted: 14
